# 05 - Entraînement et évaluation sur GPU (Kaggle)

Ce notebook relance la comparaison U-Net 2D / Wavelet-Guided U-Net sur un GPU Kaggle, avec
**tous les cas annotés** de la séparation au niveau patient (212 en entraînement, 45 en
validation, 46 en test) au lieu des 40 / 8 / 20 cas utilisés sur CPU, et **trois seeds** par
modèle au lieu de deux.

Les architectures, la perte, l'optimiseur et les augmentations sont exactement ceux des
expériences CPU (`configs/base_2d.yaml`). Le fichier `configs/kaggle/gpu_2d.yaml` ne change que la
quantité de données, la durée d'entraînement et le nombre de processus de chargement, de la même
façon pour les trois modèles. Les 20 volumes de test CPU font partie des 46 volumes de test GPU.

Déroulement : vérification du GPU, récupération du code, installation, téléchargement des
données, prétraitement, entraînements, évaluation (images originales et dégradées), analyse et
export. Une session Kaggle étant limitée à 12 h, `scripts/run_kaggle.py` applique un budget de
temps : un entraînement qui n'a plus le temps de finir s'arrête en gardant son meilleur modèle,
et l'évaluation garde tous les volumes déjà traités.

Paramètres Kaggle : accélérateur GPU, accès Internet activé (compte vérifié par téléphone).

Ce notebook a été exécuté le 26/10/2026 sur Kaggle (Tesla T4, 5,9 h). Kaggle ne renvoie pas le
notebook exécuté mais les fichiers produits : les résultats sont dans `results/kaggle/`, les
historiques d'entraînement dans `experiments/kaggle/` et le journal de l'exécution dans
`results/kaggle/journal_execution_kaggle.txt`. Ils sont commentés dans le README et dans
`docs/rapport_experiences.md` (section 8).

In [ ]:
REPO = "https://github.com/HaoudarNouhaila/wavelet-hepatic-vessel-segmentation.git"
BRANCH = "main"
SEEDS = "0 1 2"
BUDGET_HOURS = 10.8        # budget de scripts/run_kaggle.py (session de 12 h, téléchargement inclus)
TRAIN_BUDGET_HOURS = 6.3   # fin des entraînements 2D ; le U-Net 3D peut utiliser 1 h de plus

In [ ]:
!nvidia-smi
import torch, os
print("torch", torch.__version__, "| CUDA :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "pas de GPU")
!df -h /kaggle/working /tmp | cat
!nproc; free -g | head -2

## 1. Code et dépendances

In [ ]:
PROJECT = "/kaggle/working/project"
if not os.path.exists(PROJECT):
    !git clone -q --depth 1 -b {BRANCH} {REPO} {PROJECT}
%cd {PROJECT}
!git log --oneline -1
!pip install -q monai==1.6.1 PyWavelets==1.8.0 nibabel==5.4.2 SimpleITK==2.5.6
!python scripts/check_environment.py
!python -m pytest -q -x 2>&1 | tail -3

## 2. Données

L'archive officielle du Medical Segmentation Decathlon (9,3 Go, licence CC BY-SA 4.0) est
téléchargée et seuls `imagesTr` et `labelsTr` sont extraits. Les données sont placées dans `/tmp`
pour ne pas remplir le dossier de sortie de Kaggle (limité à 20 Go). La séparation au niveau
patient est recréée à l'identique (seed 42) à partir des fichiers extraits.

In [ ]:
DATA = "/tmp/whvs_data"
os.makedirs(f"{DATA}/raw", exist_ok=True)
if not os.path.exists("data"):
    os.symlink(DATA, "data")
if not os.path.exists("data/raw/Task08_HepaticVessel/labelsTr"):
    !curl -s https://msd-for-monai.s3-us-west-2.amazonaws.com/Task08_HepaticVessel.tar | tar -x -C data/raw --exclude='*/imagesTs/*' --exclude='*/._*'
!ls data/raw/Task08_HepaticVessel/imagesTr | wc -l
!ls data/raw/Task08_HepaticVessel/labelsTr | wc -l

In [ ]:
!python scripts/run_kaggle.py --steps data --workers 4

## 3. Entraînements

Ordre : seed 0 des trois modèles, puis seed 1, puis seed 2, et enfin le U-Net 3D s'il reste du
temps. Les entraînements déjà terminés sont sautés si la cellule est relancée.

In [ ]:
!python scripts/run_kaggle.py --steps train --seeds {SEEDS} --budget-hours {BUDGET_HOURS} --train-budget-hours {TRAIN_BUDGET_HOURS}

In [ ]:
!cat experiments/kaggle/run_kaggle.log | tail -20

## 4. Évaluation sur les 46 volumes de test

Images originales et sous-échantillonnées (x2, x4) pour tous les modèles ; flou de 1 mm et
sous-échantillonnage x2 avec bruit pour la seed 0. Une condition par processus.

In [ ]:
!python scripts/run_kaggle.py --steps evaluate --budget-hours {BUDGET_HOURS}

## 5. Analyse

In [ ]:
!python scripts/run_kaggle.py --steps analyze
import pandas as pd
from IPython.display import Image, display
t = pd.read_csv("results/kaggle/metrics/test_per_seed.csv")
v = t[(t["class"] == "vessel") & (t["condition"] == "original")]
display(v[["model", "seed", "dice", "cldice", "hd95_mm", "precision", "recall"]].round(4))
w = pd.read_csv("results/kaggle/metrics/test_wilcoxon_vs_unet2d.csv")
display(w[(w["condition"] == "original") & w["metric"].isin(["dice", "cldice", "hd95_mm"])].round(4))
for name in ("comparison/metriques_par_seed.png", "resolution/metriques_selon_resolution_par_seed.png",
             "training/courbes_apprentissage.png"):
    path = f"results/kaggle/figures/{name}"
    if os.path.exists(path):
        display(Image(path))

## 6. Export

Les résultats (`results/kaggle`) et les dossiers d'entraînement (configurations, historiques,
meilleurs modèles) sont regroupés dans `resultats_kaggle.zip`, téléchargeable dans l'onglet
Output du notebook. Les données et les prédictions, volumineuses, ne sont pas exportées.

In [ ]:
!zip -q -r /kaggle/working/resultats_kaggle.zip results/kaggle experiments/kaggle -x "*/last.pt"
!ls -lh /kaggle/working/resultats_kaggle.zip